# Testing Pose TCN processor

In [ ]:
import torch

print(torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import imageio.v3 as iio
import numpy as np
from PIL import Image, ImageDraw

from aitraf.processing import load_target_label_mappings
from aitraf.pose_tcn.data import PoseTCNDataset
from aitraf.pose_tcn.processing import process_sample, build_collate
from aitraf.utils import (
    get_video_rotation_deg,
    draw_pose_keypoints,
    POSE_DEFAULT_SKELETON,
)

#### Load dataset and label mappings

In [ ]:
POSES_DIR = Path("../data/poses")
CLIPS_DIR = Path("../data/clips")
MANIFESTS_DIR = Path("../data/manifests")
SPLIT = "train"
SAMPLE_FRAMES = 5
SAMPLE_COUNT = 5

dataset = PoseTCNDataset(
    manifests_dir=MANIFESTS_DIR,
    poses_dir=POSES_DIR,
    split=SPLIT,
)
_, label2id, id2label = load_target_label_mappings(MANIFESTS_DIR)
SAMPLE_INDICES = list(range(min(SAMPLE_COUNT, len(dataset))))
len(dataset)

#### Inspect processed tensors

In [ ]:
sample_row = dataset[SAMPLE_INDICES[0]]
processed_sample = process_sample(
    sample_row,
    num_frames=SAMPLE_FRAMES,
    sampling_dist="gaussian_stochastic",
    label2id=label2id,
)


def describe(batch):
    return {
        "inputs_shape": tuple(batch["inputs"].shape),
        "poses_shape": tuple(batch["poses"].shape),
        "scores_shape": tuple(batch["scores"].shape),
        "frames": batch["frames"].tolist(),
        "label_id": batch["labels"].item(),
    }


describe(processed_sample)

#### Batch using the collate function

In [ ]:
collate = build_collate(
    num_frames=SAMPLE_FRAMES,
    sampling_dist="gaussian_stochastic",
    label2id=label2id,
)

batch = collate([dataset[idx] for idx in SAMPLE_INDICES])
{k: tuple(v.shape) for k, v in batch.items()}

#### Visualize sampled poses over video frames

In [ ]:
fig, axes = plt.subplots(
    len(SAMPLE_INDICES),
    SAMPLE_FRAMES,
    figsize=(SAMPLE_FRAMES * 2.4, len(SAMPLE_INDICES) * 2.6),
)
axes = np.atleast_2d(axes)


def _load_frames(video_path: Path) -> list[np.ndarray]:
    rotation = get_video_rotation_deg(video_path) // 90
    return [np.rot90(frame, k=rotation) for frame in iio.imiter(str(video_path))]


poses_batch = batch["poses"]
frames_batch = batch["frames"]

for row, dataset_idx in enumerate(SAMPLE_INDICES):
    sample = dataset[int(dataset_idx)]
    video_path = CLIPS_DIR / sample["video_id"]
    frames = _load_frames(video_path)
    pose_tensor = poses_batch[row]
    frame_ids = frames_batch[row]

    for col, (pose, frame_idx) in enumerate(zip(pose_tensor, frame_ids)):
        frame_idx = int(frame_idx)
        frame = frames[frame_idx]
        h, w = frame.shape[:2]
        img = Image.fromarray(frame.copy())
        draw = ImageDraw.Draw(img)
        draw_pose_keypoints(
            draw, [pose], width=w, height=h, skeleton=POSE_DEFAULT_SKELETON
        )

        ax = axes[row, col]
        ax.imshow(img)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(f"frame {frame_idx}", fontsize=10)
        if col == 0:
            ax.set_ylabel(
                f"idx {dataset_idx}",
                rotation=0,
                ha="right",
                va="center",
                fontsize=10,
                labelpad=4,
            )

plt.tight_layout()
plt.show()